# Bangla Hallucination Detection — v9.2 "The Judge"

**v9.1 hotfix:** first full run crashed at the NLI cell — vLLM's install upgrades Kaggle's CUDA
python stack to 13.x, and DeBERTa's TorchScript-fused GPU kernel then fails with
`nvrtc: failed to open libnvrtc-builtins.so.13.0`. Fixed by disabling TorchScript GPU fusion
(Section 0 config) + a CPU fallback inside the NLI loop.

**v9.2 hotfix:** second run crashed importing `sentence_transformers` (its new release imports
`torchcodec`, which can't load against vLLM's torch build). Dependency dropped — e5 embeddings are
now computed with plain `transformers` + mean pooling (identical vectors). Judge/vLLM code unchanged.

**Why v1–v8 plateaued at ~0.65:** the test set is ~46% closed-book rows (`[NULL]` context).
NLI, embeddings, BanglaBERT and hand features can only measure *response ↔ context/prompt consistency* —
they contain **no world knowledge**, so on closed-book rows they are barely better than a coin flip.
The only knowledge-bearing component so far was Qwen2.5-**1.5B**, which is far too weak for Bengali facts.

**v9 strategy — one big lever plus the proven stack:**

1. **Primary signal: a large open-weight LLM judge** — `Qwen2.5-32B-Instruct-AWQ` served by **vLLM
   with tensor-parallel 2 across both T4s** (auto-fallback ladder → 14B → 7B if init fails).
   Verdicts are scored from **first-token Yes/No logprobs** (deterministic, calibrated-ish, 1 token
   generated per prompt so the run is prefill-bound and fast).
2. **Three prompt variants** (fact-check framing, hallucination framing with flipped polarity,
   Bengali-instruction framing) → self-consistency by averaging → 4 judge features.
3. **Proven cheap features kept:** mDeBERTa NLI (ent/neu/contra), multilingual-e5 similarities,
   number-match + overlap hand features.
4. **Stacking with honest nested CV** on the 299 labeled rows; candidate selection
   (judge-only vs LR stack vs blends) and per-group (`has_context` vs closed-book) thresholds are all
   chosen *inside* the inner folds — the printed nested score is the number to trust.
5. **Guarded, class-balanced pseudo-labeling** (the v3 failure mode is explicitly prevented:
   equal-count sampling per class, one round only, 50/50 blend with the pre-pseudo probabilities).

**Runtime budget (T4×2, ~9h limit):** installs ~10 min · 32B weight download+load ~25 min ·
judge over (299+2516)×3 prompts ≈ 2.5–4h · NLI+embeddings ~15 min · stacking ~5 min.
Comfortable margin; judge scores are checkpointed to disk every chunk, so a restart resumes.

**Honesty note:** nothing guarantees 0.9+. The judge attacks the exact failure mode that capped
v8, and 32B-class judges typically land 0.78–0.88 on tasks like this — where it tops out depends
on how adversarial the hidden negatives are. Trust the nested-CV number this notebook prints.


## 0. Setup

vLLM ships its own CUDA/torch stack — install it first and let it pin `transformers`.
Everything else installs without `-U` so it can't break vLLM's pins.

In [ ]:
%%time
!pip install -q vllm
!pip install -q xgboost accelerate bitsandbytes
!pip install -q git+https://github.com/csebuetnlp/normalizer


In [ ]:
import os, re, gc, json, math, time, glob, hashlib, random, warnings
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.metrics import f1_score, confusion_matrix, classification_report
import xgboost as xgb

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# --- v9.1 fix: disable TorchScript GPU kernel fusion. vLLM's install brings a CUDA-13
# python stack onto Kaggle's image, and NVRTC then fails to find libnvrtc-builtins.so.13.0
# when DeBERTa's scripted ops try to JIT-fuse a kernel ("nvrtc: error: failed to open ...").
# Eager kernels don't need NVRTC, so turning the fusers off avoids the crash entirely.
os.environ.setdefault('PYTORCH_NVFUSER_DISABLE', 'fallback')
for _fn, _arg in [('_jit_set_texpr_fuser_enabled', False),
                  ('_jit_override_can_fuse_on_gpu', False),
                  ('_jit_override_can_fuse_on_cpu', False),
                  ('_jit_set_nvfuser_enabled', False),
                  ('_jit_set_profiling_executor', True)]:
    try:
        getattr(torch._C, _fn)(_arg)
    except Exception:
        pass

# ---------------- paths (auto-discover on Kaggle, fallback to local layout) ----------------
def find_file(name_part):
    for root in ['/kaggle/input', '.', '..']:
        hits = glob.glob(os.path.join(root, '**', '*' + name_part + '*'), recursive=True)
        hits = [h for h in hits if os.path.isfile(h)]
        if hits:
            return sorted(hits, key=len)[0]
    return None

DATA_PATH = find_file('dataset samples.json') or find_file('dataset_samples.json')
TEST_PATH = find_file('test set.csv') or find_file('test_set.csv') or find_file('test.csv')
print('train:', DATA_PATH)
print('test :', TEST_PATH)
assert DATA_PATH and TEST_PATH, 'Could not locate data files — attach the competition dataset.'

WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else './v9_work'
os.makedirs(WORK, exist_ok=True)

# ---------------- feature toggles ----------------
ENABLE_JUDGE   = True    # the whole point of v9
ENABLE_NLI     = True
ENABLE_EMB     = True
ENABLE_PSEUDO  = True    # guarded — see Section 8
N_FOLDS        = 5

# ---------------- judge config ----------------
# (model, tensor_parallel, max_model_len) — tried top to bottom until one initializes
JUDGE_LADDER = [
    ('Qwen/Qwen2.5-32B-Instruct-AWQ', 2, 2048),
    ('Qwen/Qwen2.5-14B-Instruct-AWQ', 2, 3072),
    ('Qwen/Qwen2.5-14B-Instruct-AWQ', 1, 2560),
    ('Qwen/Qwen2.5-7B-Instruct-AWQ',  1, 3072),
]
JUDGE_FALLBACK_BNB = 'Qwen/Qwen2.5-14B-Instruct'   # transformers+4bit path if vLLM breaks entirely
JUDGE_CHUNK = 512                                   # prompts per vLLM call (checkpoint granularity)
CTX_CLIP, PROMPT_CLIP, RESP_CLIP = 1500, 400, 700   # char clips before token-level guard

NLI_MODEL = 'MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7'
EMB_MODEL = 'intfloat/multilingual-e5-base'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| gpus:', torch.cuda.device_count())


## 1. Load & Normalize Data

In [ ]:
try:
    from normalizer import normalize
except Exception:
    print('normalizer unavailable — falling back to identity')
    normalize = lambda s: s

NO_CTX = {'', 'nan', 'NaN', '[NULL]', 'null', 'NULL', None}

def clean(text):
    if pd.isna(text) or str(text).strip() in NO_CTX:
        return ''
    t = str(text).strip()
    try:
        return normalize(t)
    except Exception:
        return t

with open(DATA_PATH, encoding='utf-8') as f:
    df = pd.DataFrame(json.load(f))
test_df = pd.read_csv(TEST_PATH)
if 'id' not in test_df.columns:
    test_df.insert(0, 'id', np.arange(1, len(test_df) + 1))

for d in (df, test_df):
    for col in ['context', 'prompt_bn', 'response_bn']:
        d[col] = d[col].apply(clean)
    d['has_context'] = d['context'].str.len() > 0
    d['premise'] = d['context'].where(d['has_context'], d['prompt_bn'])

y = df['label'].values.astype(int)
print(f'train {len(df)}  (label1={y.sum()}, label0={(1-y).sum()}, with-context={df.has_context.sum()})')
print(f'test  {len(test_df)}  (with-context={test_df.has_context.sum()})')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
strat_key = y * 2 + df['has_context'].astype(int).values   # stratify on label x context jointly
fold_splits = list(skf.split(df, strat_key))


## 2. The LLM Judge — prompt design

Three variants per row. Instructions in English (strong models read Bengali fine; single-token
`Yes`/`No` heads are far more reliable in English), content untouched Bengali. Variant B asks the
*inverse* question — its probability is flipped — which decorrelates errors. Variant C gives
Bengali instructions to catch rows where the framing language shifts the verdict.

Few-shot exemplars (2 faithful + 2 hallucinated, shortest closed-book rows) are inlined **only for
closed-book prompts** — context rows stay zero-shot to keep prefill cheap. Exemplar rows are
**excluded from CV** so their inflated judge scores can't leak into calibration.

In [ ]:
def _clip(s, n):
    s = s or ''
    return s if len(s) <= n else s[:n] + '…'

# ---- pick 4 short closed-book exemplars (2 per class) for few-shot ----
_nc = df[~df.has_context].copy()
_nc['tot_len'] = _nc.prompt_bn.str.len() + _nc.response_bn.str.len()
FEWSHOT_IDX = (list(_nc[_nc.label == 1].sort_values('tot_len').index[:2]) +
               list(_nc[_nc.label == 0].sort_values('tot_len').index[:2]))
print('few-shot exemplar rows (excluded from CV):', FEWSHOT_IDX)

def fewshot_block():
    lines = []
    for i in FEWSHOT_IDX:
        r = df.loc[i]
        verdict = 'Yes' if r.label == 1 else 'No'
        lines.append(f"Question: {_clip(r.prompt_bn, 200)}\nProposed answer: {_clip(r.response_bn, 200)}\nCorrect? {verdict}\n")
    return '\n'.join(lines)

FEWSHOT = fewshot_block()

SYS = ('You are a meticulous bilingual fact-checker for Bengali (Bangla) text. '
       'You always answer with exactly one word: Yes or No.')

def build_prompt(ctx, q, resp, variant):
    ctx, q, resp = _clip(ctx, CTX_CLIP), _clip(q, PROMPT_CLIP), _clip(resp, RESP_CLIP)
    has_ctx = len(ctx) > 0
    if variant == 'A':   # fact-check framing; Yes = faithful
        if has_ctx:
            u = (f'Passage (Bengali):\n{ctx}\n\nQuestion (Bengali): {q}\n'
                 f'Response (Bengali): {resp}\n\n'
                 'Is the response factually correct AND fully supported by the passage, with no '
                 'fabricated or contradicting details? Answer Yes or No.')
        else:
            u = ('Here are solved examples:\n\n' + FEWSHOT +
                 f'\nNow judge this one.\nQuestion: {q}\nProposed answer: {resp}\n'
                 'Is the proposed answer factually correct? Answer Yes or No.\nCorrect?')
    elif variant == 'B':  # hallucination framing; Yes = hallucinated (flip later)
        if has_ctx:
            u = (f'Passage (Bengali):\n{ctx}\n\nQuestion (Bengali): {q}\n'
                 f'Response (Bengali): {resp}\n\n'
                 'Does the response contain ANY hallucination — information that is fabricated, '
                 'factually wrong, or not supported by the passage? Answer Yes or No.')
        else:
            u = (f'Question (Bengali): {q}\nResponse (Bengali): {resp}\n\n'
                 'Does the response contain ANY hallucination — fabricated or factually incorrect '
                 'information? Answer Yes or No.')
    else:                 # 'C' — Bengali instructions; Yes = faithful
        if has_ctx:
            u = (f'অনুচ্ছেদ:\n{ctx}\n\nপ্রশ্ন: {q}\nউত্তর: {resp}\n\n'
                 'উত্তরটি কি তথ্যগতভাবে সঠিক এবং অনুচ্ছেদ দ্বারা সমর্থিত? '
                 "শুধুমাত্র ইংরেজিতে 'Yes' অথবা 'No' লিখুন।")
        else:
            u = (f'প্রশ্ন: {q}\nউত্তর: {resp}\n\n'
                 'উত্তরটি কি তথ্যগতভাবে সঠিক? '
                 "শুধুমাত্র ইংরেজিতে 'Yes' অথবা 'No' লিখুন।")
    return u

VARIANTS = ['A', 'B', 'C']
FLIP = {'A': False, 'B': True, 'C': False}


## 3. Judge engine — vLLM with fallback ladder

Tries 32B-AWQ TP2 first; on any init failure falls down the ladder. If vLLM itself is broken on
the image, a transformers+bitsandbytes 4-bit path takes over (slower but same outputs).
Scores = P(Yes) / (P(Yes)+P(No)) from the first generated token's top-20 logprobs.

In [ ]:
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

JUDGE_BACKEND, JUDGE_NAME, llm, judge_tok = None, None, None, None

if ENABLE_JUDGE:
    n_gpu = max(torch.cuda.device_count(), 1)
    try:
        from vllm import LLM, SamplingParams
        from transformers import AutoTokenizer
        for name, tp, ml in JUDGE_LADDER:
            if tp > n_gpu:
                continue
            try:
                print(f'>>> trying {name} (tp={tp}, max_len={ml}) ...')
                llm = LLM(model=name, tensor_parallel_size=tp, dtype='half',
                          quantization='awq', max_model_len=ml,
                          gpu_memory_utilization=0.92, enforce_eager=True,
                          swap_space=2, disable_log_stats=True)
                judge_tok = AutoTokenizer.from_pretrained(name)
                JUDGE_BACKEND, JUDGE_NAME, JUDGE_MAXLEN = 'vllm', name, ml
                print(f'>>> judge online: {name}')
                break
            except Exception as e:
                print(f'    failed: {type(e).__name__}: {str(e)[:300]}')
                llm = None
                gc.collect(); torch.cuda.empty_cache()
    except Exception as e:
        print(f'vLLM import failed entirely: {e}')

    if JUDGE_BACKEND is None:
        print('>>> falling back to transformers + bitsandbytes 4-bit')
        from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
        judge_tok = AutoTokenizer.from_pretrained(JUDGE_FALLBACK_BNB)
        if judge_tok.pad_token is None:
            judge_tok.pad_token = judge_tok.eos_token
        _bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                  bnb_4bit_quant_type='nf4')
        bnb_model = AutoModelForCausalLM.from_pretrained(
            JUDGE_FALLBACK_BNB, quantization_config=_bnb, device_map='auto')
        bnb_model.eval()
        JUDGE_BACKEND, JUDGE_NAME, JUDGE_MAXLEN = 'bnb', JUDGE_FALLBACK_BNB, 2560


In [ ]:
YES_SET = {'yes', 'y', 'হ্যাঁ', 'হা', 'true'}
NO_SET  = {'no', 'n', 'না', 'false'}

def to_chat(user_msg):
    msgs = [{'role': 'system', 'content': SYS}, {'role': 'user', 'content': user_msg}]
    return judge_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def token_guard(user_msg, ctx, q, resp, variant):
    # re-clip context if the tokenized prompt would overflow max_model_len
    txt = to_chat(user_msg)
    n = len(judge_tok.encode(txt))
    while n > JUDGE_MAXLEN - 8 and len(ctx) > 200:
        ctx = ctx[:int(len(ctx) * 0.8)]
        user_msg = build_prompt(ctx, q, resp, variant)
        txt = to_chat(user_msg)
        n = len(judge_tok.encode(txt))
    if n > JUDGE_MAXLEN - 8:   # still too long (huge response) — hard truncate
        ids = judge_tok.encode(txt)[: JUDGE_MAXLEN - 8]
        txt = judge_tok.decode(ids)
    return txt

def _score_from_top(tok_prob_pairs):
    p_yes = p_no = 0.0
    for tok_str, prob in tok_prob_pairs:
        s = (tok_str or '').strip().lower()
        if s in YES_SET:
            p_yes += prob
        elif s in NO_SET:
            p_no += prob
    return p_yes / (p_yes + p_no) if (p_yes + p_no) > 1e-9 else 0.5

def judge_score_batch(chat_prompts):
    # returns list of P(yes) in input order
    if JUDGE_BACKEND == 'vllm':
        sp = SamplingParams(temperature=0.0, max_tokens=1, logprobs=20)
        outs = llm.generate(chat_prompts, sp)
        scores = []
        for o in outs:
            pairs = []
            lp0 = o.outputs[0].logprobs
            if lp0:
                for tid, lp in lp0[0].items():
                    tok_str = lp.decoded_token if lp.decoded_token is not None else judge_tok.decode([tid])
                    pairs.append((tok_str, math.exp(lp.logprob)))
            scores.append(_score_from_top(pairs))
        return scores
    # ---- bnb fallback: batched forward, read last-position logits ----
    scores = []
    B = 8
    yes_ids = [judge_tok.encode(w, add_special_tokens=False)[0] for w in ['Yes', 'yes', ' Yes']]
    no_ids  = [judge_tok.encode(w, add_special_tokens=False)[0] for w in ['No', 'no', ' No']]
    for s0 in range(0, len(chat_prompts), B):
        batch = chat_prompts[s0:s0 + B]
        enc = judge_tok(batch, return_tensors='pt', padding=True, truncation=True,
                        max_length=JUDGE_MAXLEN).to(bnb_model.device)
        with torch.no_grad():
            logits = bnb_model(**enc).logits
        last = enc['attention_mask'].sum(1) - 1
        for bi in range(len(batch)):
            probs = torch.softmax(logits[bi, last[bi]], dim=-1)
            py = float(sum(probs[i] for i in set(yes_ids)))
            pn = float(sum(probs[i] for i in set(no_ids)))
            scores.append(py / (py + pn) if (py + pn) > 1e-9 else 0.5)
    return scores


## 4. Run the judge over train + test (checkpointed)

Every (row, variant) score is cached to `judge_cache.json` in the working dir after each chunk —
a session restart resumes instead of recomputing. Polarity of variant B is flipped here, so every
stored score means **P(faithful)**.

In [ ]:
CACHE_PATH = os.path.join(WORK, f'judge_cache_{JUDGE_NAME.replace("/", "_")}.json' if JUDGE_NAME else 'judge_cache.json')
judge_cache = {}
if os.path.exists(CACHE_PATH):
    try:
        judge_cache = json.load(open(CACHE_PATH, encoding='utf-8'))
        print(f'loaded cache: {len(judge_cache)} scores')
    except Exception:
        judge_cache = {}

def row_key(ctx, q, resp, variant):
    return hashlib.sha1(f'{variant}||{ctx}||{q}||{resp}'.encode('utf-8')).hexdigest()

def run_judge(frame, tag):
    t0 = time.time()
    for variant in VARIANTS:
        todo_idx, todo_prompts = [], []
        for i, r in frame.iterrows():
            k = row_key(r.context, r.prompt_bn, r.response_bn, variant)
            if k not in judge_cache:
                u = build_prompt(r.context, r.prompt_bn, r.response_bn, variant)
                todo_idx.append(k)
                todo_prompts.append(token_guard(u, _clip(r.context, CTX_CLIP),
                                                _clip(r.prompt_bn, PROMPT_CLIP),
                                                _clip(r.response_bn, RESP_CLIP), variant))
        print(f'[{tag} variant {variant}] {len(todo_prompts)} prompts to score '
              f'({len(frame) - len(todo_prompts)} cached)')
        for s0 in range(0, len(todo_prompts), JUDGE_CHUNK):
            chunk_keys = todo_idx[s0:s0 + JUDGE_CHUNK]
            chunk_prompts = todo_prompts[s0:s0 + JUDGE_CHUNK]
            raw = judge_score_batch(chunk_prompts)
            for k, p_yes in zip(chunk_keys, raw):
                judge_cache[k] = (1.0 - p_yes) if FLIP[variant] else p_yes
            json.dump(judge_cache, open(CACHE_PATH, 'w'))
            done = min(s0 + JUDGE_CHUNK, len(todo_prompts))
            el = time.time() - t0
            print(f'    {done}/{len(todo_prompts)}  ({el/60:.1f} min elapsed)')
    # collect into columns
    for variant in VARIANTS:
        frame[f'judge_{variant}'] = [
            judge_cache[row_key(r.context, r.prompt_bn, r.response_bn, variant)]
            for _, r in frame.iterrows()]
    frame['judge_mean'] = frame[[f'judge_{v}' for v in VARIANTS]].mean(axis=1)
    return frame

if ENABLE_JUDGE:
    df = run_judge(df, 'train')
    test_df = run_judge(test_df, 'test')
else:
    for fcol in ['judge_A', 'judge_B', 'judge_C', 'judge_mean']:
        df[fcol] = 0.5; test_df[fcol] = 0.5

# quick sanity: judge alone on train
if ENABLE_JUDGE:
    for grp, sub in df.groupby('has_context'):
        pred = (sub['judge_mean'] >= 0.5).astype(int)
        print(f"has_context={grp}: judge-alone macroF1 = "
              f"{f1_score(sub['label'], pred, average='macro'):.4f}  (n={len(sub)})")


In [ ]:
# free the judge before loading the small models
if ENABLE_JUDGE and JUDGE_BACKEND == 'vllm':
    try:
        from vllm.distributed import destroy_model_parallel, destroy_distributed_environment
        del llm
        destroy_model_parallel(); destroy_distributed_environment()
    except Exception:
        pass
elif ENABLE_JUDGE and JUDGE_BACKEND == 'bnb':
    del bnb_model
gc.collect(); torch.cuda.empty_cache()
print('judge freed; free VRAM per GPU:',
      [f'{torch.cuda.mem_get_info(i)[0]/1e9:.1f}GB' for i in range(torch.cuda.device_count())])


## 5. NLI features (mDeBERTa, proven in v4–v8)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

def add_nli(frames):
    if not ENABLE_NLI:
        for fr in frames:
            fr['nli_entail'] = 0.33; fr['nli_neutral'] = 0.33; fr['nli_contra'] = 0.33
        return
    tok = AutoTokenizer.from_pretrained(NLI_MODEL)
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            NLI_MODEL, dtype=torch.float16).to(device)
    except TypeError:  # older transformers
        model = AutoModelForSequenceClassification.from_pretrained(
            NLI_MODEL, torch_dtype=torch.float16).to(device)
    model.eval()
    nli_dev = device
    for fr in frames:
        ents, neus, cons = [], [], []
        B = 32
        for s0 in range(0, len(fr), B):
            sub = fr.iloc[s0:s0 + B]
            enc = tok(list(sub['premise']), list(sub['response_bn']),
                      truncation=True, max_length=512, padding=True,
                      return_tensors='pt').to(nli_dev)
            try:
                with torch.no_grad():
                    logits = model(**enc).logits
            except RuntimeError as e:
                # last-resort guard (e.g. leftover NVRTC/CUDA issues): finish on CPU
                print(f'GPU NLI batch failed ({str(e)[:120]}) — switching to CPU fp32')
                model = model.float().cpu(); nli_dev = 'cpu'
                enc = {k: v.cpu() for k, v in enc.items()}
                with torch.no_grad():
                    logits = model(**enc).logits
            probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
            ents += list(probs[:, 0]); neus += list(probs[:, 1]); cons += list(probs[:, 2])
        fr['nli_entail'], fr['nli_neutral'], fr['nli_contra'] = ents, neus, cons
    del model; gc.collect(); torch.cuda.empty_cache()

add_nli([df, test_df])
print('NLI done')


## 6. Embedding similarity + hand-crafted features (proven set, trimmed)

In [ ]:
# v9.2: e5 embeddings via plain transformers (mean pooling) — the new sentence-transformers
# release drags in torchcodec, which fails to load against vLLM's torch build on Kaggle.
if ENABLE_EMB:
    from transformers import AutoTokenizer as _ATok, AutoModel as _AModel
    etok = _ATok.from_pretrained(EMB_MODEL)
    try:
        emodel = _AModel.from_pretrained(EMB_MODEL, dtype=torch.float16).to(device)
    except TypeError:
        emodel = _AModel.from_pretrained(EMB_MODEL, torch_dtype=torch.float16).to(device)
    emodel.eval()
    emb_dev = device

    def e5_encode(texts, B=64):
        global emodel, emb_dev
        out = []
        for s0 in range(0, len(texts), B):
            enc = etok(texts[s0:s0 + B], truncation=True, max_length=512,
                       padding=True, return_tensors='pt').to(emb_dev)
            try:
                with torch.no_grad():
                    h = emodel(**enc).last_hidden_state
            except RuntimeError as e:
                print(f'GPU emb batch failed ({str(e)[:120]}) — switching to CPU fp32')
                emodel = emodel.float().cpu(); emb_dev = 'cpu'
                enc = {k: v.cpu() for k, v in enc.items()}
                with torch.no_grad():
                    h = emodel(**enc).last_hidden_state
            mask = enc['attention_mask'].unsqueeze(-1).to(h.dtype)
            v = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-6)
            v = torch.nn.functional.normalize(v.float(), dim=-1)
            out.append(v.cpu().numpy())
        return np.vstack(out)

    def add_sims(fr):
        e_r = e5_encode(['query: ' + t for t in fr['response_bn']])
        e_p = e5_encode(['passage: ' + t for t in fr['premise']])
        e_q = e5_encode(['query: ' + t for t in fr['prompt_bn']])
        fr['sim_ctx_resp'] = np.sum(e_r * e_p, axis=1)
        fr['sim_prompt_resp'] = np.sum(e_r * e_q, axis=1)

    add_sims(df); add_sims(test_df)
    del emodel; gc.collect(); torch.cuda.empty_cache()
else:
    for fr in (df, test_df):
        fr['sim_ctx_resp'] = 0.0; fr['sim_prompt_resp'] = 0.0

BN_DIGITS = str.maketrans('০১২৩৪৫৬৭৮৯', '0123456789')
NUM_RE = re.compile(r'[০-৯0-9]+(?:[\.,][০-৯0-9]+)?')

def extract_numbers(text):
    out = set()
    for m in NUM_RE.findall(text):
        try:
            out.add(float(m.translate(BN_DIGITS).replace(',', '')))
        except Exception:
            pass
    return out

def tokens(t):
    return set(re.findall(r'[ঀ-৿a-zA-Z0-9]+', t))

def add_hand(fr):
    feats = {k: [] for k in ['resp_len', 'ctx_len', 'len_ratio', 'jaccard_cr', 'resp_new_ratio',
                             'num_match', 'num_extra', 'resp_one_word', 'has_ctx_f']}
    for _, r in fr.iterrows():
        rt, pt = tokens(r.response_bn), tokens(r.premise)
        inter, union = len(rt & pt), max(len(rt | pt), 1)
        rn, pn = extract_numbers(r.response_bn), extract_numbers(r.premise)
        feats['resp_len'].append(len(r.response_bn))
        feats['ctx_len'].append(len(r.context))
        feats['len_ratio'].append(len(r.response_bn) / max(len(r.premise), 1))
        feats['jaccard_cr'].append(inter / union)
        feats['resp_new_ratio'].append(len(rt - pt) / max(len(rt), 1))
        feats['num_match'].append(len(rn & pn) / max(len(rn), 1) if rn else -1.0)
        feats['num_extra'].append(len(rn - pn))
        feats['resp_one_word'].append(float(len(rt) <= 1))
        feats['has_ctx_f'].append(float(r.has_context))
    for k, v in feats.items():
        fr[k] = v

add_hand(df); add_hand(test_df)
HAND_COLS = ['resp_len', 'ctx_len', 'len_ratio', 'jaccard_cr', 'resp_new_ratio',
             'num_match', 'num_extra', 'resp_one_word', 'has_ctx_f']
print('features done')


## 7. Stacking with honest nested CV

Candidate deciders (per outer fold, everything selected on inner folds only):

- **judge-only** — `judge_mean` with per-group thresholds
- **LR stack** — logistic regression on all features
- **XGB stack**
- **LR+judge blend** (0.6/0.4) and **LR+XGB+judge blend** (0.4/0.2/0.4)

The winner by nested score is refit on all 295 non-exemplar rows for deployment.
Per-group thresholds (`has_context` vs closed-book) are grid-searched jointly for macro F1.

In [ ]:
FEATURE_COLS = (['judge_A', 'judge_B', 'judge_C', 'judge_mean',
                 'nli_entail', 'nli_neutral', 'nli_contra',
                 'sim_ctx_resp', 'sim_prompt_resp'] + HAND_COLS)

cv_mask = ~df.index.isin(FEWSHOT_IDX)          # exemplar rows excluded from CV
cvdf = df[cv_mask].reset_index(drop=True)
Xcv = cvdf[FEATURE_COLS].values.astype(float)
ycv = cvdf['label'].values.astype(int)
gcv = cvdf['has_context'].values
Xte = test_df[FEATURE_COLS].values.astype(float)

strat2 = ycv * 2 + gcv.astype(int)
fold_splits = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                                   random_state=SEED).split(Xcv, strat2))

def make_lr():
    return SKPipeline([('sc', StandardScaler()),
                       ('lr', LogisticRegression(C=0.3, max_iter=2000,
                                                 class_weight='balanced', random_state=SEED))])
def make_xgb():
    return xgb.XGBClassifier(n_estimators=250, max_depth=3, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0,
                             eval_metric='logloss', random_state=SEED)

# out-of-fold probabilities for the two learners
oof_lr = np.zeros(len(cvdf)); oof_xgb = np.zeros(len(cvdf))
lr_models, xgb_models = [], []
for tr, va in fold_splits:
    m1, m2 = make_lr(), make_xgb()
    m1.fit(Xcv[tr], ycv[tr]); m2.fit(Xcv[tr], ycv[tr])
    oof_lr[va] = m1.predict_proba(Xcv[va])[:, 1]
    oof_xgb[va] = m2.predict_proba(Xcv[va])[:, 1]
    lr_models.append(m1); xgb_models.append(m2)

judge_cv = cvdf['judge_mean'].values
CANDIDATES = {
    'judge_only': judge_cv,
    'lr':         oof_lr,
    'xgb':        oof_xgb,
    'lr_judge':   0.6 * oof_lr + 0.4 * judge_cv,
    'lr_xgb_judge': 0.4 * oof_lr + 0.2 * oof_xgb + 0.4 * judge_cv,
}

THR_GRID = np.arange(0.20, 0.81, 0.02)

def best_group_thresholds(scores, labels, groups, idx):
    s, l, g = scores[idx], labels[idx], groups[idx]
    best = (0.5, 0.5, -1)
    for t1 in THR_GRID:            # threshold for has_context rows
        for t0 in THR_GRID:        # threshold for closed-book rows
            pred = np.where(g, s >= t1, s >= t0).astype(int)
            f1 = f1_score(l, pred, average='macro')
            if f1 > best[2]:
                best = (t1, t0, f1)
    return best

def apply_thr(scores, groups, t1, t0):
    return np.where(groups, scores >= t1, scores >= t0).astype(int)

# ---- nested evaluation: pick candidate + thresholds on inner folds, score on outer ----
nested_scores, picked = [], []
for fold, (tr, va) in enumerate(fold_splits):
    best_cand, best_inner, best_t = None, -1, (0.5, 0.5)
    for cname, sc in CANDIDATES.items():
        t1, t0, f1 = best_group_thresholds(sc, ycv, gcv, tr)
        if f1 > best_inner:
            best_cand, best_inner, best_t = cname, f1, (t1, t0)
    pred_va = apply_thr(CANDIDATES[best_cand][va], gcv[va], *best_t)
    f1_va = f1_score(ycv[va], pred_va, average='macro')
    nested_scores.append(f1_va); picked.append(best_cand)
    print(f'outer fold {fold}: picked={best_cand:<13} inner={best_inner:.4f} '
          f'thr(ctx={best_t[0]:.2f}, nc={best_t[1]:.2f}) -> held-out={f1_va:.4f}')

print(f'\n=== HONEST NESTED-CV MACRO-F1: {np.mean(nested_scores):.4f} '
      f'± {np.std(nested_scores):.4f} ===  (this is the number to trust)')


In [ ]:
# ---- deployment: pick the candidate that won most outer folds (ties -> best mean OOF F1) ----
from collections import Counter
vote = Counter(picked)
DEPLOY_CAND = vote.most_common(1)[0][0]
t1, t0, oof_f1 = best_group_thresholds(CANDIDATES[DEPLOY_CAND], ycv, gcv, np.arange(len(cvdf)))
print(f'deployment decider: {DEPLOY_CAND}  thresholds: ctx={t1:.2f} closed-book={t0:.2f} '
      f'(in-sample OOF F1={oof_f1:.4f} — optimistic, see nested number above)')

# test-set probabilities for each candidate
lr_te  = np.mean([m.predict_proba(Xte)[:, 1] for m in lr_models], axis=0)
xgb_te = np.mean([m.predict_proba(Xte)[:, 1] for m in xgb_models], axis=0)
judge_te = test_df['judge_mean'].values
TEST_SCORES = {
    'judge_only': judge_te,
    'lr': lr_te,
    'xgb': xgb_te,
    'lr_judge': 0.6 * lr_te + 0.4 * judge_te,
    'lr_xgb_judge': 0.4 * lr_te + 0.2 * xgb_te + 0.4 * judge_te,
}
test_prob = TEST_SCORES[DEPLOY_CAND].copy()


## 8. Guarded pseudo-labeling (one round, class-balanced)

v3's failure was unbalanced pseudo-labels amplifying the majority class. Guards here:
equal counts per class (capped at 400 each), confidence ≥0.92 / ≤0.08 only, a single round,
and the refreshed probabilities are **blended 50/50** with the pre-pseudo ones rather than
replacing them. If either class has <30 confident rows, the step is skipped.

In [ ]:
if ENABLE_PSEUDO:
    hi = np.where(test_prob >= 0.92)[0]
    lo = np.where(test_prob <= 0.08)[0]
    n_take = min(len(hi), len(lo), 400)
    if n_take >= 30:
        rng = np.random.RandomState(SEED)
        hi = rng.choice(hi, n_take, replace=False)
        lo = rng.choice(lo, n_take, replace=False)
        X_aug = np.vstack([Xcv, Xte[hi], Xte[lo]])
        y_aug = np.concatenate([ycv, np.ones(n_take, int), np.zeros(n_take, int)])
        m1, m2 = make_lr(), make_xgb()
        m1.fit(X_aug, y_aug); m2.fit(X_aug, y_aug)
        lr_te2, xgb_te2 = m1.predict_proba(Xte)[:, 1], m2.predict_proba(Xte)[:, 1]
        refreshed = {
            'judge_only': judge_te, 'lr': lr_te2, 'xgb': xgb_te2,
            'lr_judge': 0.6 * lr_te2 + 0.4 * judge_te,
            'lr_xgb_judge': 0.4 * lr_te2 + 0.2 * xgb_te2 + 0.4 * judge_te,
        }[DEPLOY_CAND]
        test_prob = 0.5 * test_prob + 0.5 * refreshed
        print(f'pseudo-labeling applied: +{n_take} per class')
    else:
        print(f'pseudo-labeling skipped (confident rows: hi={len(hi)}, lo={len(lo)})')


## 9. Submission

In [ ]:
test_pred = apply_thr(test_prob, test_df['has_context'].values, t1, t0)

sub = pd.DataFrame({'id': test_df['id'].values, 'label': test_pred})
SUB_PATH = os.path.join(WORK, 'submission.csv')
sub.to_csv(SUB_PATH, index=False)

print(f'wrote {SUB_PATH}  ({len(sub)} rows)')
print('\nprediction distribution:')
print(sub['label'].value_counts(normalize=True).rename({0: 'hallucinated(0)', 1: 'faithful(1)'}))
print('\nby group:')
for grp, s in sub.groupby(test_df['has_context'].values)['label']:
    print(f'  has_context={grp}: mean(label)={s.mean():.3f}  n={len(s)}')
print('\nSanity: train is 45% label-0 / 55% label-1. If the test prediction is >70% one class, '
      'something is off — check the judge sanity print in Section 4 before submitting.')

# keep features for reuse / ablations in v9.x
try:
    df.to_parquet(os.path.join(WORK, 'train_features_v9.parquet'))
    test_df.to_parquet(os.path.join(WORK, 'test_features_v9.parquet'))
except Exception as e:
    df.to_csv(os.path.join(WORK, 'train_features_v9.csv'), index=False)
    test_df.to_csv(os.path.join(WORK, 'test_features_v9.csv'), index=False)


## Summary (for the report / 7-min video)

**Pipeline:** Qwen2.5-32B-Instruct-AWQ judge (vLLM, TP2, first-token Yes/No logprobs, 3 prompt
variants with one flipped polarity) → +mDeBERTa NLI, e5 similarities, number/overlap hand features →
LR/XGB stack with nested-CV candidate & per-group threshold selection → guarded balanced
pseudo-labeling → submission.

**Why it works:** ~46% of the test set is closed-book — consistency features carry no world
knowledge there. A 32B judge does. The stack keeps the context rows at v8-level or better while the
judge lifts the closed-book half from near-coin-flip.

**Rulebook compliance:** open-weight models only (Qwen — Apache 2.0 / Qwen license, mDeBERTa MIT,
e5 MIT), all inference local in this Kaggle notebook, no external APIs, no external data beyond
public model weights.

**Knobs if you're short on time:** switch `JUDGE_LADDER` to start at 14B (≈2× faster, usually
1–3 F1 points cheaper) · drop variant C (`VARIANTS = ['A','B']`) · `ENABLE_PSEUDO=False`.

**If the 32B fails to load on your session:** the ladder handles it automatically; nothing to do.
Check the Section 3 log to see which judge actually served.
